# 🚀 Colab RAG GPU Server

This notebook loads the `SentenceTransformer` (`all-MiniLM-L6-v2`) model onto a **Google Colab GPU** and exposes high-performance embedding & semantic retrieval endpoints over FastAPI + ngrok for your local AI Voice Assistant.

### Features:
- ⚡ **CUDA GPU Accelerated**: Embedding computation in milliseconds.
- 📡 **FastAPI Endpoints**: `/health`, `/embed`, `/search`, `/rerank`.
- 🌐 **ngrok Public Tunnel**: Instant connectivity to local assistant.

In [ ]:
import torch

print("=== Google Colab GPU Verification ===")
print("CUDA Available :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Count   :", torch.cuda.device_count())
    print("Device Name    :", torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"VRAM Total     : {total_mem:.2f} GB")
else:
    print("WARNING: CUDA is not available. Please change runtime to GPU!")

In [ ]:
!pip install -q sentence-transformers fastapi uvicorn pydantic pyngrok nest_asyncio requests python-multipart

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

MODEL_NAME = "all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[SERVER] Loading SentenceTransformer model '{MODEL_NAME}' on device: {device}...")

model = SentenceTransformer(MODEL_NAME, device=device)
print(f"[SERVER] Model '{MODEL_NAME}' loaded successfully on {device}!")

In [ ]:
import time
from typing import List, Optional
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException
import numpy as np

app = FastAPI(title="Colab GPU RAG Retrieval Server", version="1.0.0")

class EmbedRequest(BaseModel):
    text: Optional[str] = None
    texts: Optional[List[str]] = None

class SearchRequest(BaseModel):
    query: str
    documents: List[str]
    top_n: Optional[int] = 5

class RerankRequest(BaseModel):
    query: str
    documents: List[str]

@app.get("/health")
def health_check():
    gpu_info = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_alloc = torch.cuda.memory_allocated(0) / (1024**2) if torch.cuda.is_available() else 0
    return {
        "status": "healthy",
        "model": MODEL_NAME,
        "device": device,
        "gpu": gpu_info,
        "vram_allocated_mb": round(vram_alloc, 2),
        "timestamp": time.time()
    }

@app.post("/embed")
def generate_embeddings_api(req: EmbedRequest):
    t0 = time.perf_counter()
    if req.text:
        vec = model.encode(req.text, convert_to_numpy=True, normalize_embeddings=True)
        t_infer = time.perf_counter() - t0
        return {
            "embedding": vec.tolist(),
            "shape": list(vec.shape),
            "inference_time_ms": round(t_infer * 1000, 2)
        }
    elif req.texts:
        vecs = model.encode(req.texts, batch_size=32, convert_to_numpy=True, normalize_embeddings=True)
        t_infer = time.perf_counter() - t0
        return {
            "embeddings": vecs.tolist(),
            "count": len(vecs),
            "inference_time_ms": round(t_infer * 1000, 2)
        }
    else:
        raise HTTPException(status_code=400, detail="Must provide 'text' or 'texts'.")

@app.post("/search")
def search_api(req: SearchRequest):
    if not req.documents:
        return {"results": [], "query": req.query}
        
    t0 = time.perf_counter()
    q_vec = model.encode(req.query, convert_to_numpy=True, normalize_embeddings=True)
    doc_vecs = model.encode(req.documents, batch_size=32, convert_to_numpy=True, normalize_embeddings=True)
    
    similarities = np.dot(doc_vecs, q_vec)
    ranked_indices = np.argsort(similarities)[::-1][:req.top_n]
    
    results = []
    for idx in ranked_indices:
        results.append({
            "index": int(idx),
            "document": req.documents[idx],
            "score": float(similarities[idx])
        })
        
    t_infer = time.perf_counter() - t0
    return {
        "results": results,
        "query": req.query,
        "inference_time_ms": round(t_infer * 1000, 2)
    }

@app.post("/rerank")
def rerank_api(req: RerankRequest):
    search_res = search_api(SearchRequest(query=req.query, documents=req.documents, top_n=len(req.documents)))
    return {"reranked": search_res["results"], "query": req.query}

In [ ]:
import os
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import threading

# Allow nested asyncio loops in Colab
nest_asyncio.apply()

# Open HTTP tunnel on port 8000
public_url = ngrok.connect(8000).public_url
print("=" * 60)
print(f"🚀 Colab RAG GPU Server is online!")
print(f"🔗 Public API URL: {public_url}")
print(f"📋 Health Endpoint: {public_url}/health")
print(f"📋 Embed Endpoint : {public_url}/embed")
print(f"📋 Search Endpoint: {public_url}/search")
print("=" * 60)
print("Copy public_url into your local assistant .env file: RAG_API_URL=" + public_url)

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

In [ ]:
import requests
import time

time.sleep(2)  # Wait for uvicorn server thread

print("Testing /health endpoint...")
h_res = requests.get("http://localhost:8000/health").json()
print("Health Response:", h_res)

print("\nTesting /embed endpoint...")
e_res = requests.post("http://localhost:8000/embed", json={"text": "Open HealthSphere document"}).json()
print("Embed Response Shape:", e_res["shape"], "Latency:", e_res["inference_time_ms"], "ms")

print("\nTesting /search endpoint...")
s_res = requests.post("http://localhost:8000/search", json={
    "query": "HealthSphere",
    "documents": ["HealthSphere design spec.pdf", "Money Mentor user guide.docx", "Random notes.txt"]
}).json()
print("Search Results:", s_res["results"])
print("\nAll endpoints verified successfully!")